**Set environment**

In [1]:
import numpy  as np
import pandas as pd
import itertools as it
from functools import partial
import os, sys, re
import csv

In [2]:
%run ../run_config_project.py
show_env()

BASE DIRECTORY (FD_BASE): /hpc/group/igvf/kk319
REPO DIRECTORY (FD_REPO): /hpc/group/igvf/kk319/repo
WORK DIRECTORY (FD_WORK): /hpc/group/igvf/kk319/work
DATA DIRECTORY (FD_DATA): /hpc/group/igvf/kk319/data


You are working with      IGVF BlueSTARR
PATH OF PROJECT (FD_PRJ): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR
PROJECT RESULTS (FD_RES): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results
PROJECT SCRIPTS (FD_EXE): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/scripts
PROJECT DATA    (FD_DAT): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/data
PROJECT NOTE    (FD_NBK): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/notebooks
PROJECT DOCS    (FD_DOC): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/docs
PROJECT LOG     (FD_LOG): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/log
PROJECT REF     (FD_REF): /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/references



In [3]:
FP_GEN = "/hpc/group/igvf/kk319/data/genome/hg38/hg38.fa"

import pysam
fasta  = pysam.FastaFile(FP_GEN)

#from pyfaidx import Fasta
#fasta  = Fasta(FP_GEN)

## Import data

In [4]:
### set file directory
txt_fdiry = os.path.join(FD_RES, "analysis_variant_motif_richard")
txt_fname = "variant_closed_gof_bluestarr.tsv"
txt_fpath = os.path.join(txt_fdiry, txt_fname)

### read table
dat = pd.read_csv(txt_fpath, sep = "\t")

### assign and show
dat_variant_import = dat
print(dat.shape)
dat.head()

(1885038, 15)


,Chrom,ChromStart,ChromEnd,Region,Variant_Ref,Variant_UnObs_vs_Obs,Pos0,Rank4,Rank3,Rank2,Rank1,Ref,Obs,Unobs,Delta
0,chr2,234143174,234143434,chr2:234143174-234143434,chr2:234143291:C,chr2:234143291:C:C:G,234143291,C,T,A,G,C,C,G,1.708901
1,chr6,111030999,111031188,chr6:111030999-111031188,chr6:111031101:A,chr6:111031101:A:G:C,111031101,G,A,T,C,A,G,C,1.625281
2,chr5,73091238,73091414,chr5:73091238-73091414,chr5:73091405:T,chr5:73091405:T:C:A,73091405,C,T,G,A,T,C,A,1.597880
3,chr2,37426938,37427245,chr2:37426938-37427245,chr2:37427187:G,chr2:37427187:G:G:T,37427187,G,A,C,T,G,G,T,1.526402
4,chr6,27903836,27904383,chr6:27903836-27904383,chr6:27904376:G,chr6:27904376:G:C:A,27904376,C,G,T,A,G,C,A,1.502800


## Sanity check: reference allele

Check if the reference allele column is correct

**Check the first row**

In [5]:
txt_region = "chr4:74487576-74488141"

txt_chrom_name, txt_chrom_slice = txt_region.split(":")
txt_chrom_name  = str(txt_chrom_name)
txt_chrom_pos   = 74487586

print(f"Chrom: {txt_chrom_name}; Start-End: {txt_chrom_slice}; Pos: {txt_chrom_pos}")

Chrom: chr4; Start-End: 74487576-74488141; Pos: 74487586


In [6]:
### 0-based slicing
print(fasta.fetch(txt_chrom_name, txt_chrom_pos, txt_chrom_pos+1))
#print(fasta[txt_chrom_name][txt_chrom_pos:txt_chrom_pos+1].seq)

A


**Check more positions**

In [7]:
%%time
### init
dat = dat_variant_import
dat = dat.sample(1000, random_state=1)

### fetch ref allele and check mismatches
lst_mismatches = []

for txt_chrom_name, num_chrom_pos, txt_allele_ref_table in zip(
    dat["Chrom"],
    dat["Pos0"], 
    dat["Ref"]
):
    ### query the reference allele
    ### note: pos is 0-based, end-exclusive
    txt_allele_ref_fetch = fasta.fetch(txt_chrom_name, num_chrom_pos, num_chrom_pos+1) 

    ### sequence has uppercase and lowercase bases
    ### - uppercase: high-confidence sequence
    ### - lowercase: soft-masked sequence
    ### convert all to uppercase for simplicity
    txt_allele_ref_fetch = txt_allele_ref_fetch.upper()
    
    ### check match/mismatch
    if txt_allele_ref_fetch != txt_allele_ref_table:
        tmp = (txt_chrom_name, num_chrom_pos, txt_allele_ref_table, txt_allele_ref_fetch)
        lst_mismatches.append(tmp)

print(f"Checked {len(dat)} variants")
print(f"Found {len(lst_mismatches)} mismatches")
if lst_mismatches:
    print("Example mismatches:", lst_mismatches[:5])

Checked 1000 variants
Found 0 mismatches
CPU times: user 27.7 ms, sys: 18.5 ms, total: 46.2 ms
Wall time: 45 ms


## Helper function

In [8]:
def get_interval_refseq(
    txt_chrom_name, 
    num_chrom_pos0, 
    txt_allele_ref,
    num_interval_flank_left, 
    num_interval_flank_right
):
    ### fetch genomic interval centered at variant position
    txt_seq_ref = fasta.fetch(
        txt_chrom_name,
        num_chrom_pos0 - num_interval_flank_left,
        num_chrom_pos0 + num_interval_flank_right + 1
    ).upper()

    ### sanity check reference allele
    if txt_seq_ref[num_interval_flank_left].upper() != txt_allele_ref.upper():
        raise ValueError(
            f"Reference mismatch at {txt_chrom_name}:{txt_chrom_pos0} "
            f"(expected {txt_allele_ref}, got {txt_seq_ref[num_interval_flank_left]})"
        )

    ### return sequence
    return txt_seq_ref

## Get sequence by specifying flanking size as L35bp R70bp

In [9]:
### spanning x bp at the center of each variant
### here the flanking region is set as 35 bp at both side
NUM_INTERVAL_FLANK_LEFT  = 35 
NUM_INTERVAL_FLANK_RIGHT = 70 

**Test run**

In [10]:
dat.iloc[0,:]

Chrom                                     chr17
ChromStart                             40604941
ChromEnd                               40605046
Region                  chr17:40604941-40605046
Variant_Ref                    chr17:40605021:C
Variant_UnObs_vs_Obs       chr17:40605021:C:C:G
Pos0                                   40605021
Rank4                                         C
Rank3                                         T
Rank2                                         A
Rank1                                         G
Ref                                           C
Obs                                           C
Unobs                                         G
Delta                                  0.012019
Name: 1562725, dtype: object

In [11]:
%%time

### init: get random 100 variants
dat = dat_variant_import.copy()
dat = dat.sample(100, random_state=123)
lst_txt_seq = []

### add sequences
fun = partial(
    get_interval_refseq, 
    num_interval_flank_left  = NUM_INTERVAL_FLANK_LEFT,
    num_interval_flank_right = NUM_INTERVAL_FLANK_RIGHT
)

### loop through each variant and get refseq
for txt_chrom_name, num_chrom_pos0, txt_allele_ref in zip(
    dat["Chrom"],
    dat["Pos0"],
    dat["Ref"]
):
    txt_seq = fun(
        txt_chrom_name,
        int(num_chrom_pos0),
        txt_allele_ref
    )
    lst_txt_seq.append(txt_seq)

### assign and show
dat["Seq_Ref"] = lst_txt_seq
dat.head()

CPU times: user 173 ms, sys: 22.3 ms, total: 196 ms
Wall time: 195 ms


,Chrom,ChromStart,ChromEnd,Region,Variant_Ref,Variant_UnObs_vs_Obs,Pos0,Rank4,Rank3,Rank2,Rank1,Ref,Obs,Unobs,Delta,Seq_Ref
1168372,chr16,11749963,11750093,chr16:11749963-11750093,chr16:11750040:G,chr16:11750040:G:C:T,11750040,C,G,A,T,G,C,T,0.019044,TGCCCCCGACACATGGAGGGAACCAGACTCTCCTAGTATTTTTAGA...
858914,chr12,132382369,132383145,chr12:132382369-132383145,chr12:132383142:C,chr12:132383142:C:T:A,132383142,T,C,G,A,C,T,A,0.025211,GGTACACAGAAGACAGAAGACATACTCAGCTAAGGCAGTACCTGAT...
377075,chr13,57190564,57191410,chr13:57190564-57191410,chr13:57191359:G,chr13:57191359:G:G:C,57191359,G,A,T,C,G,G,C,0.041085,GGCTTCTGCTTTATGGTGCCTTTTGCAAGTTCTATGCAGCCTGCTT...
954843,chr4,42092389,42092613,chr4:42092389-42092613,chr4:42092399:T,chr4:42092399:T:A:G,42092399,A,T,C,G,T,A,G,0.023144,AAGGCAGAGTCAGGATTTGAACTCACAAGGAAGCATTCTGCTAAAC...
1851931,chr1,111115523,111115769,chr1:111115523-111115769,chr1:111115612:A,chr1:111115612:A:C:T,111115612,C,A,G,T,A,C,T,0.004644,TAAAAGTTAAATATTCAACTCAAAAATGTTGATGGAATTGAAGAGA...


## Export fasta file

**Helper function**

In [12]:
def fun_wrap_fasta(seq, width=60):
    """
    Wrap a sequence string into fixed-width lines for FASTA output.
    """
    return "\n".join(
        seq[i:i + width]
        for i in range(0, len(seq), width)
    )

### Export: Unobserved vs Observed

In [13]:
dat = dat_variant_import.copy()
#dat["Variant_ID"] = dat["Variant_ID"] + ":" + dat["Obs"] + ":" + dat["Unobs"]
dat["Variant_ID"] = dat["Variant_UnObs_vs_Obs"]
dat_variant_arrange = dat
dat.head(3)

,Chrom,ChromStart,ChromEnd,Region,Variant_Ref,Variant_UnObs_vs_Obs,Pos0,Rank4,Rank3,Rank2,Rank1,Ref,Obs,Unobs,Delta,Variant_ID
0,chr2,234143174,234143434,chr2:234143174-234143434,chr2:234143291:C,chr2:234143291:C:C:G,234143291,C,T,A,G,C,C,G,1.708901,chr2:234143291:C:C:G
1,chr6,111030999,111031188,chr6:111030999-111031188,chr6:111031101:A,chr6:111031101:A:G:C,111031101,G,A,T,C,A,G,C,1.625281,chr6:111031101:A:G:C
2,chr5,73091238,73091414,chr5:73091238-73091414,chr5:73091405:T,chr5:73091405:T:C:A,73091405,C,T,G,A,T,C,A,1.597880,chr5:73091405:T:C:A


In [14]:
%%time

### set file directory
txt_fdiry = os.path.join(FD_RES, "analysis_variant_motif_richard")
txt_fname = "variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.ref.fa"
#txt_fname = "variant_closed_gof_bluestarr.flankL35R70.rank1_vs_obs.ref.fa"
#txt_fname = "variant_closed_gof_bluestarr.flankL35R70.rank2_vs_obs.ref.fa"
txt_fpath = os.path.join(txt_fdiry, txt_fname)

### init: function
fun = partial(
    get_interval_refseq, 
    num_interval_flank_left  = NUM_INTERVAL_FLANK_LEFT,
    num_interval_flank_right = NUM_INTERVAL_FLANK_RIGHT
)

### loop through each variant and get refseq
with open(txt_fpath, "w") as fout:
    for idx, (txt_chrom_name, num_chrom_pos0, txt_allele_ref, txt_variant_idx) in enumerate(
        zip(
            dat_variant_arrange["Chrom"],
            dat_variant_arrange["Pos0"],
            dat_variant_arrange["Ref"],
            dat_variant_arrange["Variant_ID"],
        ),
        start=1
    ):
        ### verbose progress
        if idx % 100_000 == 0:
            print(f"Written {idx:,} sequences...")

        ### get refseq
        txt_seq = fun(
            txt_chrom_name,
            int(num_chrom_pos0),
            txt_allele_ref
        )
        
        ### output refseq
        fout.write(f">{txt_variant_idx}\n")
        fout.write(fun_wrap_fasta(txt_seq, width=60) + "\n")

print("Wrote:", txt_fpath)

Written 100,000 sequences...
Written 200,000 sequences...
Written 300,000 sequences...
Written 400,000 sequences...
Written 500,000 sequences...
Written 600,000 sequences...
Written 700,000 sequences...
Written 800,000 sequences...
Written 900,000 sequences...
Written 1,000,000 sequences...
Written 1,100,000 sequences...
Written 1,200,000 sequences...
Written 1,300,000 sequences...
Written 1,400,000 sequences...
Written 1,500,000 sequences...
Written 1,600,000 sequences...
Written 1,700,000 sequences...
Written 1,800,000 sequences...
Wrote: /hpc/group/igvf/kk319/repo/Proj_IGVF_BlueSTARR/results/analysis_variant_motif_richard/variant_closed_gof_bluestarr.flankL35R70.unobs_vs_obs.ref.fa
CPU times: user 7.51 s, sys: 34.8 s, total: 42.3 s
Wall time: 2min 1s
